# 0027 / 03 Canonical features

Compile the winner-only rows with the vendored 0025 causal knowledge tracker and canonical feature compiler.


In [ ]:
from __future__ import annotations
import gzip, importlib, json, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/ptcg_0027_canonical_features")
OUT.mkdir(parents=True, exist_ok=True)
SCHEMA_VERSION = "0025_canonical_semantic_decision_v2"
source_candidates = sorted({
    path.parent.parent
    for path in INPUT.rglob("official_public_prototypes_v1.json")
    if path.parent.name == "assets" and (path.parent.parent / "model" / "canonical").is_dir()
})
if len(source_candidates) != 1:
    raise FileNotFoundError("attach the vendored 0025 source package")
source_root = source_candidates[0]
sys.path.insert(0, str(source_root.parent))
package = source_root.name
prototypes_mod = importlib.import_module(f"{package}.features.prototypes")
compiler_mod = importlib.import_module(f"{package}.features.canonical.compiler")
knowledge_mod = importlib.import_module(f"{package}.knowledge.state")
PrototypeIndex, compile_canonical_row, CausalKnowledge = (
    prototypes_mod.PrototypeIndex, compiler_mod.compile_canonical_row, knowledge_mod.CausalKnowledge
)
prototypes = PrototypeIndex.load(source_root / "assets" / "official_public_prototypes_v1.json")
raw_candidates = sorted(INPUT.rglob("winner_raw.jsonl.gz"))
if len(raw_candidates) != 1:
    raise FileNotFoundError(f"expected exactly one notebook 02 winner slice, found {len(raw_candidates)}")
raw_path = raw_candidates[0]

def registered_deck(row):
    counts = row.get("deck_manifest", {}).get("counts")
    if not isinstance(counts, list):
        raise ValueError("winner row has no canonical deck counts")
    deck = [int(identity) for identity, count in counts for _ in range(int(count))]
    if len(deck) != 60:
        raise ValueError("winner row deck manifest is not exactly 60 cards")
    return deck

records = 0
group_key = None
knowledge = None
maximum_lengths = {}
with gzip.open(raw_path, "rt", encoding="utf-8") as source, gzip.open(OUT / "canonical_records.jsonl.gz", "wt", encoding="utf-8") as target:
    for line in source:
        raw = json.loads(line)
        identity = raw["identity"]
        key = (identity["date"], identity["episode_id"], int(identity["player_index"]))
        if key != group_key:
            group_key = key
            knowledge = CausalKnowledge(key[2], registered_deck(raw))
        assert knowledge is not None
        snapshot = knowledge.consume(raw["actor_observation"], raw.get("event_cursor"))
        compiled = compile_canonical_row(raw, snapshot, prototypes)
        actor = compiled["actor"]
        for name in ("card_cat", "resource_cat", "event_cat", "option_cat", "option_skill_id", "option_effect_id"):
            maximum_lengths[name] = max(maximum_lengths.get(name, 0), len(actor[name]))
        record = {"schema_version": SCHEMA_VERSION, "actor": actor, "target": compiled["target"], "audit": {"identity": identity, "split": raw["split"], "source_id": raw.get("source_id"), "source_payload_sha256": raw.get("source_payload_sha256", "")}}
        target.write(json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n")
        records += 1

(OUT / "feature_compile_manifest.json").write_text(json.dumps({
    "schema_version": SCHEMA_VERSION,
    "records": records,
    "maximum_lengths": maximum_lengths,
    "compiler": "0025.features.canonical.compiler.compile_canonical_row",
    "actor_forward_excludes": ["legacy", "ordered_action", "source_id", "source_team_name", "source_payload_sha256"],
}, indent=2) + "\n", encoding="utf-8")
print(json.dumps({"records": records, "output": str(OUT), "maximum_lengths": maximum_lengths}, indent=2))

